# Soil Fertility Index (SFI) — Data Pipeline

This notebook does five things, in order:

1. **Merge** the five yearly soil-test CSV files into one dataset
2. **Clean** the merged data (fix errors, drop incomplete rows, harmonise N)
3. **Normalise** each of the 12 soil parameters to a 0–1 score
4. **Compute SFI** (Soil Fertility Index) as a weighted sum, and assign a **Low / Medium / High** class
5. **Export** the final table as a downloadable CSV

Run the cells from top to bottom. Each section has a short explanation before the code.


## 0. Setup

Import the libraries we need. `pandas` does almost everything here; `re` is used once, to clean up messy column headers.

In [2]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 20)


ModuleNotFoundError: No module named 'pandas'

### File locations

Edit the `SRC` folder path and the five filenames below so they match where your CSV files are saved.
If your notebook is in the same folder as the CSV files, leave `SRC` as an empty string.


In [ ]:
SRC = "C:/Users/athre/Desktop/Fertilizer - MiniProject/DataSets"   

files = {
    "2019-20":       "2019-20_Soil_Test_Results_Samples-2500_Samples.csv",
    "2019-20_Pilot": "2019-20_Soil_Test_Results_Samples_PILOT_SCHEME_3_cycle.csv",
    "2020-21":       "2020-21_Soil_Test_Results_Samples.csv",
    "2022-23":       "2022-23_Soil_Test_Results_Samples.csv",
    "2024-25":       "2024-25_Soil_Test_Results_Samples.csv",
}


## 1. Combine the 5 yearly datasets

The five files don't share identical column names — for example, pH is written `PH` in some files and `pH` in
others, and EC appears as `EC` in some and `EC       ds/m` (extra spaces included) in others. Before we can
stack the files on top of each other, every file needs the **same 12 column names**.

`standardize_column()` below does this: it collapses extra whitespace, then looks at just the *first word* of
each header (`"EC"` out of `"EC       ds/m"`, `"N"` out of `"N          kg/hectre"`) and maps it to one clean
name.

We also:
- **Rename the first column to `Sample_ID`** — every file has a per-row ID column, but it's called something
  different in each file (`Reg Sl. No`, `Register No`, `Sl. No.`).
- **Drop any "Unnamed" columns** — the 2020-21 file has 13 blank trailing columns pandas will otherwise load in.
- **Add a `Year` column** — so we can always trace a row back to which file it came from.


In [ ]:
# Maps the first word of a column header (lowercased) to one standard name
STD_MAP = {
    'ph': 'pH', 'ec': 'EC', 'oc': 'OC', 'n': 'N', 'p': 'P', 'k': 'K', 's': 'S',
    'zn': 'Zn', 'b': 'B', 'fe': 'Fe', 'mn': 'Mn', 'cu': 'Cu',
}

def standardize_column(col):
    """'EC       ds/m' -> 'EC' ; 'N          kg/hectre' -> 'N' ; 'PH' -> 'pH' """
    cleaned = re.sub(r"\s+", " ", str(col)).strip()   # collapse multiple spaces into one
    first_word = cleaned.split(" ")[0].lower()
    return STD_MAP.get(first_word, cleaned)             # fall back to the cleaned name if not a known parameter


In [ ]:
frames = []

for year, filename in files.items():
    df = pd.read_csv(SRC + filename, encoding="utf-8-sig")   # utf-8-sig strips a hidden BOM character some files start with

    df.columns = [standardize_column(c) for c in df.columns]
    df = df.loc[:, [c for c in df.columns if not c.lower().startswith("unnamed")]]  # drop blank trailing columns (2020-21 file)
    df = df.rename(columns={df.columns[0]: "Sample_ID"})     # first column is always the row ID, whatever it's called

    df["Year"] = year
    frames.append(df)

    print(f"{year:16s} -> {df.shape[0]:5d} rows, {df.shape[1]} columns")

merged = pd.concat(frames, ignore_index=True)
print(f"\nTotal merged rows: {len(merged)}")


**Check:** the total should be exactly `18702` (2500 + 1090 + 4562 + 5208 + 5342). If it's different, a file didn't load, or was loaded twice.

In [ ]:
assert len(merged) == 18702, f"Expected 18702 rows, got {len(merged)} — check the file paths above."
merged.head()


## 2. Clean the merged data

Three real problems exist in this data, found by inspecting it directly — not assumptions:

1. **One impossible pH value.** A single row has `pH = 534`, almost certainly a decimal-point typo for `5.34`.
   We detect any pH above 14 (the maximum possible on the pH scale) and divide it by 100.
2. **Missing / invalid values.** A few rows have blank cells or an Excel error (`#REF!`) in place of a number.
   These rows are dropped, since there's no reliable value to use.
3. **Nitrogen (N) is not independently measured — it's calculated from OC**, and the lab used a *different*
   multiplier in different years:

   | Year | N ÷ OC |
   |---|---|
   | 2019-20 / 2019-20 Pilot | 74.1 |
   | 2020-21 | 74.1 or 97.8 (mixed) |
   | 2022-23 | 97.8 (mostly) |
   | 2024-25 | 224.0 (mostly) |

   This means raw N values **aren't comparable across years** — the same soil would get a different N value
   depending only on which year it was tested. We fix this by recomputing N for every row using one fixed
   multiplier (74.1, the one used in the earliest files): `N = OC × 74.1`. The original reported value is kept
   in a separate `N_raw_reported` column, in case you want to compare or use it instead.


In [ ]:
PARAMS = ['pH', 'EC', 'OC', 'N', 'P', 'K', 'S', 'Zn', 'B', 'Fe', 'Mn', 'Cu']

# Force every parameter column to numeric. Anything that can't convert (e.g. "#REF!") becomes NaN (missing).
for c in PARAMS:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")


In [ ]:
# --- Fix 1: impossible pH values (pH can never exceed 14) ---
bad_ph_mask = merged["pH"] > 14
print(f"pH values above 14 found: {bad_ph_mask.sum()}")
merged.loc[bad_ph_mask, "pH"] = merged.loc[bad_ph_mask, "pH"] / 100


In [ ]:
# --- Fix 2: drop rows with any missing/invalid parameter value ---
rows_before = len(merged)
merged = merged.dropna(subset=PARAMS).reset_index(drop=True)
rows_dropped = rows_before - len(merged)
print(f"Rows dropped (missing or invalid values): {rows_dropped}")
print(f"Rows remaining: {len(merged)}")


In [ ]:
# --- Fix 3: harmonise N onto one common basis across all years ---
N_FACTOR = 74.1   # the multiplier used in the earliest (2019-20) files

merged["N_raw_reported"] = merged["N"]          # keep the original value, for reference / comparison
merged["N"] = merged["OC"] * N_FACTOR           # recompute N so every year uses the same formula

merged[["Year", "OC", "N_raw_reported", "N"]].head()


**Check:** after these three fixes, you should have `18668` rows left (18702 − 34 dropped).

In [ ]:
assert len(merged) == 18668, f"Expected 18668 rows after cleaning, got {len(merged)}"


## 3. Normalise each of the 12 parameters (0–1 scale)

The 12 parameters are in different units (pH has no unit, K is kg/ha, B is ppm), so they can't be combined
directly. Each is rescaled to a **0 to 1 score**, where 1 always means "good for fertility" and 0 means "poor".

There are three different rules, because "good" means something different for each parameter:

| Rule | Used for | Logic |
|---|---|---|
| **More is better** | OC, N, P, K, S, Zn, B, Fe, Mn, Cu | Higher value = healthier soil. Score = `value / Xmax`, capped at 1 |
| **Less is better** | EC | Lower salinity = healthier soil. Score = `(Xmax − value) / Xmax` |
| **Optimal range** | pH | Best at 6.5, worse the further away. Score = `1 − \|value − 6.5\| / 3.5` |

**The `Xmax` values below are typical "sufficient" thresholds for Indian soils** (e.g. P is considered
sufficient above 56 kg/ha). They are *not* taken from your dataset — they're standard reference values, so you
should double-check them against your own source (an ICAR chart, your state soil-testing lab's rating scale,
or a paper your guide gives you) before treating them as final.


In [ ]:
# "Sufficient" upper bound (Xmax) for each "more is better" parameter. Xmin = 0 for all of them.
BOUNDS = {
    'OC': (0.0, 1.5),      # %
    'N':  (0.0, 111.15),   # kg/ha  (= 1.5 * 74.1, since N is derived from OC)
    'P':  (0.0, 56.0),     # kg/ha
    'K':  (0.0, 336.0),    # kg/ha
    'S':  (0.0, 20.0),     # ppm
    'Zn': (0.0, 1.2),      # ppm
    'B':  (0.0, 1.0),      # ppm
    'Fe': (0.0, 10.0),     # ppm
    'Mn': (0.0, 5.0),      # ppm
    'Cu': (0.0, 1.0),      # ppm
}

EC_MIN, EC_MAX = 0.0, 2.0        # dS/m — EC uses "less is better"
PH_OPTIMAL, PH_MAX_DEVIATION = 6.5, 3.5   # pH uses "optimal range"


In [ ]:
def normalize_more_is_better(value, xmin, xmax):
    """Higher value -> higher score. Used for OC, N, P, K, S, Zn, B, Fe, Mn, Cu."""
    score = (value - xmin) / (xmax - xmin)
    return score.clip(0, 1)          # cap between 0 and 1 — a value far above Xmax doesn't mean "extra fertile"

def normalize_less_is_better(value, xmin, xmax):
    """Lower value -> higher score. Used for EC."""
    score = (xmax - value) / (xmax - xmin)
    return score.clip(0, 1)

def normalize_optimal_range(value, optimal, max_deviation):
    """Closer to the optimal value -> higher score. Used for pH."""
    score = 1 - (value - optimal).abs() / max_deviation
    return score.clip(0, 1)


In [ ]:
norm = pd.DataFrame(index=merged.index)

norm["pH"] = normalize_optimal_range(merged["pH"], PH_OPTIMAL, PH_MAX_DEVIATION)
norm["EC"] = normalize_less_is_better(merged["EC"], EC_MIN, EC_MAX)

for param, (xmin, xmax) in BOUNDS.items():
    norm[param] = normalize_more_is_better(merged[param], xmin, xmax)

norm.head()


## 4. Compute the SFI value and assign a fertility class

### 4a. Weights

Not every parameter matters equally. OC, N, P, K and pH are "major indicators" and get more weight;
micronutrients (Zn, B, Fe, Mn, Cu) and EC/S get less. **These weights were assigned by hand** (following
general guidance that major indicators should dominate), not learned from the data or derived by PCA — that's
worth mentioning if you're asked how they were chosen.

The weights must add up to **1.0**.

### 4b. The SFI formula

$$SFI = \sum_{i=1}^{12} (N_i \times w_i)$$

Where $N_i$ is the normalised 0–1 score from Step 3, and $w_i$ is that parameter's weight.

### 4c. Classification

Once every sample has an SFI value, we sort samples into three classes using two cutoffs. Here the cutoffs are
the **33rd and 67th percentiles** of the SFI values themselves, so the three classes come out roughly equal in
size. (These aren't official agronomic thresholds — just a way to get a balanced Low/Medium/High split. Swap in
fixed cutoffs here if your course specifies exact SFI ranges.)


In [ ]:
WEIGHTS = {
    'OC': 0.15, 'N': 0.10, 'P': 0.10, 'K': 0.10, 'pH': 0.10,   # major indicators (sum 0.55)
    'EC': 0.05, 'S': 0.05,                                      # sum 0.10
    'Zn': 0.08, 'B': 0.08, 'Fe': 0.07, 'Mn': 0.06, 'Cu': 0.06,  # micronutrients (sum 0.35)
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"


In [ ]:
def compute_sfi(norm_row, weights=WEIGHTS):
    """
    Compute SFI for one row of normalised (0-1) parameter scores.
    norm_row: a pandas Series with one 0-1 score per parameter (pH, EC, OC, N, P, K, S, Zn, B, Fe, Mn, Cu)
    Returns: a single float, the weighted sum SFI = sum(N_i * w_i)
    """
    return sum(norm_row[param] * weights[param] for param in weights)


# Apply the function to every row
merged["SFI"] = norm.apply(compute_sfi, axis=1).round(4)

merged[["Year", "Sample_ID", "Place", "SFI"]].head()


In [ ]:
def classify_sfi(sfi_value, low_cutoff, high_cutoff):
    """Turn a numeric SFI score into a Low / Medium / High label."""
    if sfi_value < low_cutoff:
        return "Low"
    elif sfi_value < high_cutoff:
        return "Medium"
    else:
        return "High"


# Cutoffs = 33rd and 67th percentile of SFI, so the 3 classes come out roughly balanced
low_cutoff, high_cutoff = merged["SFI"].quantile([1/3, 2/3]).round(4)
print(f"Low  cutoff : SFI < {low_cutoff}")
print(f"High cutoff : SFI >= {high_cutoff}")

merged["SFI_Class"] = merged["SFI"].apply(lambda sfi: classify_sfi(sfi, low_cutoff, high_cutoff))

merged["SFI_Class"].value_counts()


### Worked check

Compare this against a value you already know: the first sample (2019-20, Shirva) should come out to **SFI = 0.7458**, class **High**.

In [ ]:
print(merged.loc[0, ["Year", "Place", "SFI", "SFI_Class"]])


## 5. Export the final dataset as a CSV

We keep the original 12 parameters, the ID/location columns, and the two new columns (`SFI`, `SFI_Class`).
The normalised 0–1 scores are also included (columns named `N_pH`, `N_EC`, etc.) so you can double-check any
row's calculation later without re-running the notebook.


In [ ]:
# Rename the normalised columns so they don't clash with the raw parameter columns, then attach them
norm_renamed = norm.add_prefix("N_")
final = pd.concat([merged, norm_renamed], axis=1)

output_columns = (
    ["Year", "Sample_ID", "Place"]
    + PARAMS
    + ["N_raw_reported", "SFI", "SFI_Class"]
    + [f"N_{p}" for p in PARAMS]
)

final = final[output_columns]
final.head()


In [ ]:
OUTPUT_FILENAME = "SFI_final_dataset.csv"
final.to_csv(OUTPUT_FILENAME, index=False)
print(f"Saved {len(final)} rows to {OUTPUT_FILENAME}")


### Downloading the file

- **Jupyter Notebook / JupyterLab (running locally):** the CSV is saved in the same folder as this notebook.
  Open the file browser panel on the left, find `SFI_final_dataset.csv`, right-click it, and choose **Download**.
- **Google Colab:** run the two lines below — they'll trigger a browser download automatically.


In [ ]:
# Uncomment and run these two lines ONLY if you're using Google Colab:

# from google.colab import files
# files.download(OUTPUT_FILENAME)


---
## Summary

| Step | Result |
|---|---|
| Merge | 18,702 rows from 5 files combined into one table |
| Clean | 1 pH value fixed, 34 incomplete rows dropped, N recomputed on a common basis → 18,668 rows |
| Normalise | Each of the 12 parameters scaled to 0–1 |
| SFI | Weighted sum of the 12 normalised scores |
| Classify | Low / Medium / High, split at the 33rd/67th percentile |
| Export | `SFI_final_dataset.csv`, ready to download |
